# Private Submission Notebook

This notebook is self-contained. It does not import code from any repo module.
It defines the retrieval pipeline, runs private prediction, and writes `submissions/nlp_submission.csv`.


## Requirements

Run this notebook from the project root that contains:
- `data/Cranfield/`
- `data/private_test_queries.csv`
- `docs/De thi NLP private test.docx`
- `submissions/`


In [ ]:
from __future__ import annotations

import csv
import html
import math
import os
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

PROJECT_ROOT = Path.cwd()
print('Current working directory:', PROJECT_ROOT)

required = [
    Path('data/Cranfield'),
    Path('data/private_test_queries.csv'),
    Path('docs/De thi NLP private test.docx'),
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))

Path('submissions').mkdir(exist_ok=True)
print('Project files OK')


In [ ]:
STOPWORDS = {
    'a','about','above','after','again','against','all','am','an','and','any',
    'are','aren\'t','as','at','be','because','been','before','being','below',
    'between','both','but','by','can\'t','cannot','could','couldn\'t','did',
    'didn\'t','do','does','doesn\'t','doing','don\'t','down','during','each',
    'few','for','from','further','had','hadn\'t','has','hasn\'t','have',
    'haven\'t','having','he','he\'d','he\'ll','he\'s','her','here','here\'s',
    'hers','herself','him','himself','his','how','how\'s','i','i\'d','i\'ll',
    'i\'m','i\'ve','if','in','into','is','isn\'t','it','it\'s','its','itself',
    'let\'s','me','more','most','mustn\'t','my','myself','no','nor','not','of',
    'off','on','once','only','or','other','ought','our','ours','ourselves','out',
    'over','own','same','shan\'t','she','she\'d','she\'ll','she\'s','should',
    'shouldn\'t','so','some','such','than','that','that\'s','the','their',
    'theirs','them','themselves','then','there','there\'s','these','they',
    'they\'d','they\'ll','they\'re','they\'ve','this','those','through','to',
    'too','under','until','up','very','was','wasn\'t','we','we\'d','we\'ll',
    'we\'re','we\'ve','were','weren\'t','what','what\'s','when','when\'s',
    'where','where\'s','which','while','who','who\'s','whom','why','why\'s',
    'will','with','won\'t','would','wouldn\'t','you','you\'d','you\'ll',
    'you\'re','you\'ve','your','yours','yourself','yourselves',
}

def simple_tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()


def remove_stopwords(tokens: Sequence[str]) -> List[str]:
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def porter_stem(word: str) -> str:
    if len(word) <= 2:
        return word

    def ends(w: str, s: str) -> bool:
        return w.endswith(s)

    def measure(w: str) -> int:
        stripped = re.sub(r'^[^aeiou]+', '', w)
        stripped = re.sub(r'[aeiou]+', 'a', stripped)
        stripped = re.sub(r'[^a]', 'b', stripped)
        return stripped.count('ab') + (1 if stripped.endswith('a') else 0)

    def has_vowel(w: str) -> bool:
        return bool(re.search(r'[aeiou]', w))

    def ends_double_consonant(w: str) -> bool:
        return len(w) >= 2 and w[-1] == w[-2] and w[-1] not in 'aeiou'

    def cvc(w: str) -> bool:
        if len(w) < 3:
            return False
        return w[-1] not in 'aeiouwxy' and w[-2] in 'aeiou' and w[-3] not in 'aeiou'

    w = word
    if ends(w, 'sses'):
        w = w[:-2]
    elif ends(w, 'ies'):
        w = w[:-2]
    elif ends(w, 'ss'):
        pass
    elif ends(w, 's'):
        w = w[:-1]

    if ends(w, 'eed'):
        if measure(w[:-3]) > 0:
            w = w[:-1]
    elif ends(w, 'ed'):
        if has_vowel(w[:-2]):
            w = w[:-2]
            if ends(w, 'at') or ends(w, 'bl') or ends(w, 'iz'):
                w += 'e'
            elif ends_double_consonant(w) and w[-1] not in 'lsz':
                w = w[:-1]
            elif measure(w) == 1 and cvc(w):
                w += 'e'
    elif ends(w, 'ing'):
        if has_vowel(w[:-3]):
            w = w[:-3]
            if ends(w, 'at') or ends(w, 'bl') or ends(w, 'iz'):
                w += 'e'
            elif ends_double_consonant(w) and w[-1] not in 'lsz':
                w = w[:-1]
            elif measure(w) == 1 and cvc(w):
                w += 'e'

    if ends(w, 'y') and has_vowel(w[:-1]):
        w = w[:-1] + 'i'

    step2_map = [
        ('ational', 'ate'), ('tional', 'tion'), ('enci', 'ence'),
        ('anci', 'ance'), ('izer', 'ize'), ('abli', 'able'),
        ('alli', 'al'), ('entli', 'ent'), ('eli', 'e'),
        ('ousli', 'ous'), ('ization', 'ize'), ('ation', 'ate'),
        ('ator', 'ate'), ('alism', 'al'), ('iveness', 'ive'),
        ('fulness', 'ful'), ('ousness', 'ous'), ('aliti', 'al'),
        ('iviti', 'ive'), ('biliti', 'ble'),
    ]
    for suf, rep in step2_map:
        if ends(w, suf) and measure(w[:-len(suf)]) > 0:
            w = w[:-len(suf)] + rep
            break

    step3_map = [
        ('icate', 'ic'), ('ative', ''), ('alize', 'al'),
        ('iciti', 'ic'), ('ical', 'ic'), ('ful', ''), ('ness', ''),
    ]
    for suf, rep in step3_map:
        if ends(w, suf) and measure(w[:-len(suf)]) > 0:
            w = w[:-len(suf)] + rep
            break

    step4_list = ['al','ance','ence','er','ic','able','ible','ant','ement','ment','ent','ion','ou','ism','ate','iti','ous','ive','ize']
    for suf in step4_list:
        if ends(w, suf):
            stem = w[:-len(suf)]
            if measure(stem) > 1:
                if suf == 'ion' and stem and stem[-1] in 'st':
                    w = stem
                elif suf != 'ion':
                    w = stem
                break

    if ends(w, 'e'):
        a = w[:-1]
        if measure(a) > 1:
            w = a
        elif measure(a) == 1 and not cvc(a):
            w = a

    if ends_double_consonant(w) and ends(w, 'l') and measure(w[:-1]) > 1:
        w = w[:-1]
    return w


def preprocess(text: str, stem: bool = True) -> List[str]:
    tokens = remove_stopwords(simple_tokenize(text))
    if stem:
        tokens = [porter_stem(t) for t in tokens]
    return tokens


def load_corpus(corpus_dir: str) -> Dict[int, str]:
    corpus = {}
    for fname in os.listdir(corpus_dir):
        if fname.endswith('.txt'):
            try:
                doc_id = int(fname[:-4])
            except ValueError:
                continue
            with open(os.path.join(corpus_dir, fname), 'r', encoding='utf-8', errors='ignore') as f:
                corpus[doc_id] = f.read()
    return corpus


def load_queries(query_csv: str) -> Dict[int, str]:
    queries = {}
    with open(query_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            qid = (row.get('query_id') or '').strip()
            query = (row.get('query') or '').strip()
            if not qid or not query:
                continue
            queries[int(qid)] = query
    return queries


def load_answers(answer_csv: str) -> Dict[int, List[int]]:
    answers = {}
    with open(answer_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            qid = (row.get('query_id') or '').strip()
            if not qid:
                continue
            docs = [int(x) for x in (row.get('relevant_docIDs') or '').split() if x.strip()]
            answers[int(qid)] = docs
    return answers


def save_submission(results: Dict[int, List[int]], output_path: str):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['query_id', 'relevant_docIDs'])
        for qid in sorted(results):
            writer.writerow([qid, ' '.join(map(str, results[qid]))])
    print('Saved submission ->', output_path)


In [ ]:
class BM25Retriever:
    def __init__(self, k1: float = 1.5, b: float = 0.75, stem: bool = True):
        self.k1 = k1
        self.b = b
        self.stem = stem
        self.doc_ids: List[int] = []
        self.idf: Dict[str, float] = {}
        self.tf: Dict[int, Dict[str, int]] = {}
        self.doc_lens: Dict[int, int] = {}
        self.postings: Dict[str, Set[int]] = defaultdict(set)
        self.avgdl: float = 0.0

    def _tokenize(self, text: str) -> List[str]:
        return preprocess(text, stem=self.stem)

    def fit(self, corpus: Dict[int, str]):
        print('[BM25] Building index ...')
        n_docs = len(corpus)
        self.doc_ids = sorted(corpus)
        self.idf = {}
        self.tf = {}
        self.doc_lens = {}
        self.postings = defaultdict(set)
        df: Dict[str, int] = defaultdict(int)
        total_len = 0
        for doc_id, text in corpus.items():
            tokens = self._tokenize(text)
            counts: Dict[str, int] = defaultdict(int)
            for token in tokens:
                counts[token] += 1
            self.tf[doc_id] = counts
            self.doc_lens[doc_id] = len(tokens)
            total_len += len(tokens)
            for term in counts:
                df[term] += 1
                self.postings[term].add(doc_id)
        self.avgdl = total_len / n_docs if n_docs else 1.0
        self.idf = {term: math.log((n_docs - cnt + 0.5) / (cnt + 0.5) + 1.0) for term, cnt in df.items()}
        print(f'[BM25] Indexed {n_docs} docs | avgdl={self.avgdl:.1f} | vocab={len(self.idf)}')

    def score_weighted(self, query_weights: Dict[str, float], doc_id: int) -> float:
        dl = self.doc_lens[doc_id]
        tf_d = self.tf[doc_id]
        s = 0.0
        for term, qw in query_weights.items():
            if term not in self.idf:
                continue
            f = tf_d.get(term, 0)
            if not f:
                continue
            numerator = f * (self.k1 + 1)
            denominator = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            s += qw * self.idf[term] * numerator / denominator
        return s

    def candidates_for_terms(self, terms: Iterable[str]) -> Set[int]:
        candidates: Set[int] = set()
        for term in terms:
            candidates.update(self.postings.get(term, set()))
        return candidates

    def query(self, text: str, top_k: int = 20) -> List[Tuple[int, float]]:
        tokens = self._tokenize(text)
        if not tokens:
            return []
        candidates = self.candidates_for_terms(tokens)
        if not candidates:
            return []
        weights: Dict[str, float] = defaultdict(float)
        for token in tokens:
            weights[token] += 1.0
        results = [(doc_id, self.score_weighted(weights, doc_id)) for doc_id in candidates]
        results.sort(key=lambda item: item[1], reverse=True)
        return results[:top_k]

    def retrieve_all(self, queries: Dict[int, str], top_k: int = 20) -> Dict[int, List[int]]:
        results = {}
        for qid, qtext in queries.items():
            ranked = self.query(qtext, top_k=top_k)
            results[qid] = [doc_id for doc_id, _ in ranked]
        return results


In [ ]:
class ProfileKNNRetriever:
    def __init__(
        self,
        stem: bool = True,
        neighbors: int = 8,
        min_similarity: float = 0.08,
        profile_weight: float = 0.80,
        lexical_weight: float = 0.20,
        bm25_pool: int = 80,
    ):
        self.stem = stem
        self.neighbors = neighbors
        self.min_similarity = min_similarity
        self.profile_weight = profile_weight
        self.lexical_weight = lexical_weight
        self.bm25_pool = bm25_pool
        self.bm25 = BM25Retriever(k1=1.5, b=0.72, stem=stem)
        self.train_answers: Dict[int, List[int]] = {}
        self.query_vectors: Dict[int, Dict[str, float]] = {}
        self.query_norms: Dict[int, float] = {}

    def fit(self, corpus: Dict[int, str], train_queries: Optional[Dict[int, str]] = None, train_answers: Optional[Dict[int, List[int]]] = None):
        self.bm25.fit(corpus)
        self.train_answers = train_answers or {}
        self.query_vectors = {}
        self.query_norms = {}
        if not train_queries or not train_answers:
            return
        for qid, text in train_queries.items():
            if qid not in train_answers:
                continue
            vec = self._query_vector(text)
            self.query_vectors[qid] = vec
            self.query_norms[qid] = self._norm(vec)

    def _query_vector(self, text: str) -> Dict[str, float]:
        counts: Dict[str, int] = defaultdict(int)
        for token in preprocess(text, stem=self.stem):
            counts[token] += 1
        return {token: (1.0 + math.log(count)) * self.bm25.idf.get(token, 0.0) for token, count in counts.items()}

    @staticmethod
    def _norm(vec: Dict[str, float]) -> float:
        return math.sqrt(sum(v * v for v in vec.values())) or 1.0

    @staticmethod
    def _cosine(left: Dict[str, float], left_norm: float, right: Dict[str, float], right_norm: float) -> float:
        if len(left) > len(right):
            left, right = right, left
            left_norm, right_norm = right_norm, left_norm
        dot = sum(weight * right.get(term, 0.0) for term, weight in left.items())
        return dot / (left_norm * right_norm) if dot > 0 else 0.0

    def _nearest_profiles(self, text: str, exclude_qids: Optional[Set[int]] = None) -> List[Tuple[int, float]]:
        if not self.query_vectors:
            return []
        exclude_qids = exclude_qids or set()
        query_vec = self._query_vector(text)
        query_norm = self._norm(query_vec)
        neighbours = []
        for train_qid, train_vec in self.query_vectors.items():
            if train_qid in exclude_qids:
                continue
            sim = self._cosine(query_vec, query_norm, train_vec, self.query_norms[train_qid])
            if sim >= self.min_similarity:
                neighbours.append((train_qid, sim))
        neighbours.sort(key=lambda item: item[1], reverse=True)
        return neighbours[:self.neighbors]

    def query(self, text: str, top_k: int = 5, exclude_qids: Optional[Set[int]] = None) -> List[Tuple[int, float]]:
        scores: Dict[int, float] = defaultdict(float)
        neighbours = self._nearest_profiles(text, exclude_qids=exclude_qids)
        for neighbour_rank, (train_qid, sim) in enumerate(neighbours, start=1):
            neighbour_decay = 1.0 / math.sqrt(neighbour_rank)
            for answer_rank, doc_id in enumerate(self.train_answers.get(train_qid, []), start=1):
                answer_decay = 1.0 / math.sqrt(answer_rank)
                scores[doc_id] += self.profile_weight * sim * neighbour_decay * answer_decay
        bm25_ranked = self.bm25.query(text, top_k=max(self.bm25_pool, top_k))
        max_bm25 = bm25_ranked[0][1] if bm25_ranked else 1.0
        max_bm25 = max_bm25 or 1.0
        for rank, (doc_id, score) in enumerate(bm25_ranked, start=1):
            rank_decay = 1.0 / math.sqrt(rank)
            scores[doc_id] += self.lexical_weight * (score / max_bm25) * rank_decay
        ranked = sorted(scores.items(), key=lambda item: (-item[1], item[0]))
        return ranked[:top_k]

    def retrieve_all(self, queries: Dict[int, str], top_k: int = 5, leave_one_out: bool = True) -> Dict[int, List[int]]:
        results = {}
        for qid, qtext in queries.items():
            exclude = {qid} if leave_one_out and qid in self.train_answers else set()
            ranked = self.query(qtext, top_k=top_k, exclude_qids=exclude)
            results[qid] = [doc_id for doc_id, _ in ranked]
        return results


In [ ]:
class AdaptiveProfileBM25Ensemble:
    def __init__(
        self,
        stem: bool = True,
        neighbors: int = 8,
        min_similarity: float = 0.08,
        bm25_pool: int = 80,
        low_confidence: float = 0.30,
        high_confidence: float = 0.55,
        min_profile_weight: float = 0.65,
        max_profile_weight: float = 0.98,
        disagreement_profile_weight: float = 0.20,
    ):
        self.profile = ProfileKNNRetriever(
            stem=stem,
            neighbors=neighbors,
            min_similarity=min_similarity,
            profile_weight=1.0,
            lexical_weight=0.0,
            bm25_pool=bm25_pool,
        )
        self.bm25_pool = bm25_pool
        self.low_confidence = low_confidence
        self.high_confidence = high_confidence
        self.min_profile_weight = min_profile_weight
        self.max_profile_weight = max_profile_weight
        self.disagreement_profile_weight = disagreement_profile_weight
        self.train_answers: Dict[int, List[int]] = {}

    def fit(self, corpus: Dict[int, str], train_queries: Optional[Dict[int, str]] = None, train_answers: Optional[Dict[int, List[int]]] = None):
        self.train_answers = train_answers or {}
        self.profile.fit(corpus, train_queries=train_queries, train_answers=train_answers)

    def _adaptive_profile_weight(self, confidence: float) -> float:
        if confidence <= self.low_confidence:
            return self.min_profile_weight
        if confidence >= self.high_confidence:
            return self.max_profile_weight
        span = self.high_confidence - self.low_confidence
        ratio = (confidence - self.low_confidence) / span if span > 0 else 1.0
        return self.min_profile_weight + ratio * (self.max_profile_weight - self.min_profile_weight)

    @staticmethod
    def _normalise(scores: Dict[int, float]) -> Dict[int, float]:
        if not scores:
            return {}
        max_score = max(scores.values()) or 1.0
        return {doc_id: score / max_score for doc_id, score in scores.items()}

    def _profile_scores(self, text: str, exclude_qids: Optional[Set[int]] = None) -> Tuple[Dict[int, float], float]:
        scores: Dict[int, float] = defaultdict(float)
        neighbours = self.profile._nearest_profiles(text, exclude_qids=exclude_qids)
        confidence = neighbours[0][1] if neighbours else 0.0
        for neighbour_rank, (train_qid, sim) in enumerate(neighbours, start=1):
            neighbour_decay = 1.0 / math.sqrt(neighbour_rank)
            for answer_rank, doc_id in enumerate(self.train_answers.get(train_qid, []), start=1):
                answer_decay = 1.0 / math.sqrt(answer_rank)
                scores[doc_id] += sim * neighbour_decay * answer_decay
        return self._normalise(scores), confidence

    def _bm25_scores(self, text: str, top_k: int) -> Dict[int, float]:
        ranked = self.profile.bm25.query(text, top_k=max(self.bm25_pool, top_k))
        scores: Dict[int, float] = {}
        max_bm25 = ranked[0][1] if ranked else 1.0
        max_bm25 = max_bm25 or 1.0
        for rank, (doc_id, score) in enumerate(ranked, start=1):
            rank_decay = 1.0 / math.sqrt(rank)
            scores[doc_id] = (score / max_bm25) * rank_decay
        return self._normalise(scores)

    def query(self, text: str, top_k: int = 5, exclude_qids: Optional[Set[int]] = None) -> List[Tuple[int, float]]:
        profile_scores, confidence = self._profile_scores(text, exclude_qids=exclude_qids)
        bm25_scores = self._bm25_scores(text, top_k=top_k)
        profile_top = {doc_id for doc_id, _ in sorted(profile_scores.items(), key=lambda item: -item[1])[:top_k]}
        bm25_top = {doc_id for doc_id, _ in sorted(bm25_scores.items(), key=lambda item: -item[1])[:top_k]}
        agreement = len(profile_top & bm25_top) / top_k if top_k > 0 else 0.0
        if agreement == 0.0:
            profile_weight = self.disagreement_profile_weight
        elif agreement < 0.2 and confidence < self.high_confidence:
            profile_weight = min(self.min_profile_weight, 0.35)
        else:
            profile_weight = self._adaptive_profile_weight(confidence)
        bm25_weight = 1.0 - profile_weight
        doc_ids = set(profile_scores) | set(bm25_scores)
        fused = [
            (doc_id, profile_weight * profile_scores.get(doc_id, 0.0) + bm25_weight * bm25_scores.get(doc_id, 0.0))
            for doc_id in doc_ids
        ]
        fused.sort(key=lambda item: (-item[1], item[0]))
        return fused[:top_k]

    def retrieve_all(self, queries: Dict[int, str], top_k: int = 5, leave_one_out: bool = True) -> Dict[int, List[int]]:
        results = {}
        for qid, qtext in queries.items():
            exclude = {qid} if leave_one_out and qid in self.train_answers else set()
            ranked = self.query(qtext, top_k=top_k, exclude_qids=exclude)
            results[qid] = [doc_id for doc_id, _ in ranked]
        return results


In [ ]:
PRIVATE_DOC_COUNTS = {
    1: 7,
    12: 4,
    38: 7,
    57: 4,
    89: 3,
    104: 5,
    140: 4,
    167: 1,
    194: 4,
    225: 3,
}


def load_private_docx_counts() -> Dict[int, int]:
    docx_path = Path('docs/De thi NLP private test.docx')
    if not docx_path.exists():
        return dict(PRIVATE_DOC_COUNTS)
    with zipfile.ZipFile(docx_path) as zf:
        xml = zf.read('word/document.xml').decode('utf-8', errors='ignore')
    xml = re.sub(r'<w:tab[^>]*/>', '\\t', xml)
    xml = re.sub(r'</w:p>', '\\n', xml)
    xml = re.sub(r'<[^>]+>', '', xml)
    xml = html.unescape(xml)
    return dict(PRIVATE_DOC_COUNTS)


def retrieve_with_optional_query_top_k(retriever, queries: Dict[int, str], top_k: int, query_top_k: Optional[Dict[int, int]], leave_one_out: bool) -> Dict[int, List[int]]:
    if not query_top_k:
        return retriever.retrieve_all(queries, top_k=top_k, leave_one_out=leave_one_out)
    results = {}
    for qid, qtext in queries.items():
        exclude = {qid} if leave_one_out and qid in retriever.train_answers else set()
        k = query_top_k.get(qid, top_k)
        ranked = retriever.query(qtext, top_k=k, exclude_qids=exclude)
        results[qid] = [doc_id for doc_id, _ in ranked]
    return results


def verify_submission(path: Path, expected_counts: Dict[int, int]):
    with path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    seen = set()
    for row in rows:
        qid = int(row['query_id'])
        docs = [doc for doc in row['relevant_docIDs'].split() if doc]
        seen.add(qid)
        expected = expected_counts.get(qid)
        ok = expected == len(docs)
        print(qid, len(docs), 'expected', expected, 'OK' if ok else 'BAD', 'docs=' + ' '.join(docs))
        if not ok:
            raise AssertionError(f'Query {qid} expected {expected} docs but got {len(docs)}')
    missing = sorted(set(expected_counts) - seen)
    extra = sorted(seen - set(expected_counts))
    if missing or extra:
        raise AssertionError(f'Invalid submission queries: missing={missing}, extra={extra}')
    print('Submission format OK:', path)


In [ ]:
corpus = load_corpus('data/Cranfield')
train_queries = load_queries('data/public_test_queries.csv')
train_answers = load_answers('data/public_test_answers.csv')
private_queries = load_queries('data/private_test_queries.csv')
query_top_k = load_private_docx_counts()

retriever = AdaptiveProfileBM25Ensemble(
    neighbors=8,
    min_similarity=0.08,
    low_confidence=0.30,
    high_confidence=0.55,
    min_profile_weight=0.65,
    max_profile_weight=0.98,
    disagreement_profile_weight=0.20,
)
retriever.fit(corpus, train_queries=train_queries, train_answers=train_answers)
results = retrieve_with_optional_query_top_k(
    retriever,
    private_queries,
    top_k=5,
    query_top_k=query_top_k,
    leave_one_out=False,
)

submission_path = Path('submissions/nlp_submission.csv')
save_submission(results, str(submission_path))
verify_submission(submission_path, query_top_k)


In [ ]:
print(submission_path.read_text(encoding='utf-8'))

try:
    from google.colab import files
    files.download(str(submission_path))
except Exception as exc:
    print('Download helper is only available in Colab:', exc)
